# Assault DDQN - HU002

Notebook scaffold for the reproducible Assault environment pipeline. This notebook intentionally validates only HU002: configuration, hardware inspection, environment creation, preprocessing, seeds, train/eval separation and short interaction smoke checks.

## 1. Dependencies

In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("2_Assault") / "requirements.txt"
%pip install -q -r {requirements_path}

## 2. Imports and configuration

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "2_Assault":
    PROJECT_DIR = PROJECT_DIR / "2_Assault"
sys.path.insert(0, str(PROJECT_DIR))

from src.environment import create_assault_env, get_environment_metadata, validate_frameskip_once
from src.utils import get_runtime_info, load_yaml_config

config = load_yaml_config(PROJECT_DIR / "configs" / "ddqn_config.yaml")
seed = int(config["reproducibility"]["seed"])
config

## 3. Runtime and hardware

In [ ]:
runtime_info = get_runtime_info()
runtime_info

## 4. Environment contract

In [ ]:
train_env = create_assault_env(config, mode="train", seed=seed)
eval_env = create_assault_env(config, mode="eval", seed=seed + 1)

obs, info = train_env.reset(seed=seed)
metadata = get_environment_metadata(train_env, config, mode="train", seed=seed)

print("Observation shape:", obs.shape)
print("Observation dtype:", obs.dtype)
print("Action space:", train_env.action_space)
print("Action meanings:", train_env.unwrapped.get_action_meanings())
print("Initial info:", info)
print("Metadata:", metadata)

## 5. HU002 autovalidations

In [ ]:
assert obs.shape == (4, 84, 84)
assert str(obs.dtype) == "uint8"
assert train_env.action_space.n == 7
assert train_env.observation_space.shape == eval_env.observation_space.shape
assert train_env.observation_space.dtype == eval_env.observation_space.dtype
assert validate_frameskip_once(train_env, expected_frameskip=4, steps=5)

obs, info = train_env.reset(seed=seed)
for step in range(100):
    action = int(train_env.action_space.sample())
    obs, reward, terminated, truncated, info = train_env.step(action)
    assert obs.shape == (4, 84, 84)
    assert str(obs.dtype) == "uint8"
    if terminated or truncated:
        obs, info = train_env.reset()

print("HU002 validations passed.")

In [ ]:
train_env.close()
eval_env.close()